<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/main/colabs/sts_demo_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Okta OAuth-STS Acces Demo

This notebook demonstrates the complete **STS** flow for accessing SAAS resources using the Okta AI SDK.


## The 3-Step OAuth STS Flow

1. **Get ID Token** - Authenticate as a user to obtain an ID Token
1. **Exchange for a Resource Token** - Exchange the user's token for stored resource access token
1. **Resource API Access** - Use the resource's access token to call the resource API.
---

### Prerequisites

Before running this notebook, you need:

1. **Okta Organization** with:
   - Resource Application server configured

2. **Agent Principal** with:
   - Principal ID (agent identifier)
   - Private JWK (RSA key pair for JWT bearer assertion)

3. **User**:
   - Valid Okta User to authenticate and obtain an ID token.

4. **Execution order for extended steps in this notebook**:
   - Run setup and Steps 1 before running Steps 2-3.
   - Step 2 expects `ID_TOKEN`, `agent_sdk`, and (optionally) `REFRESH_TOKEN` to already exist.

### Configuration Options

You can configure the SDK using either:
- **Direct values** in the code (for testing)
- **Variables** (recommended for security)

## Setup and Installation


### 1. Install the SDK

In [ ]:
# Install the Okta AI SDK from PyPI

%pip install okta-client-python

### 2. Set Configuration

Set _all_ your configuration variables and _run the cell._

In [ ]:
# @title 1. Configuration Variables { display-mode: "form" }
# @markdown _Enter your configuration variables here and click the 'run' to validate._
# @markdown <br><br> These will be used throughout the notebook.

# OKTA_DOMAIN = 'https://{your_domain}.oktapreview.com' # @param {"type":"string","placeholder":"Enter your entire Okta domain (including https)"}
# @markdown <br>
OKTA_DOMAIN = '' # @param {"type":"string","placeholder":"Enter your entire Okta domain (including https)"}
REDIRECT_URI = 'http://localhost:8080/authorization-code/callback' # @param {"type":"string","placeholder":"Enter application redirect URI"}

# @markdown <br>
# @markdown <hr>
# @markdown <br>
# @markdown <em><sub>If you created an application as part of your agent registration, do not enter a CLIENT_ID or CLIENT_SECRET.</sub></em><br>
CLIENT_ID = '' # @param {"type":"string","placeholder":"Enter your client Id"}

CLIENT_SECRET = '' # @param {"type":"string","placeholder":"Enter your client secret"}

# @markdown <br>
# @markdown <hr>
# @markdown <br>
PRINCIPAL_ID = '' # @param {"type":"string","placeholder":"Enter your agent identifier"}
PRIVATE_JWK = {} # @param {"type":"raw","placeholder": "Agent Private JWK"}

# @markdown <br>
# @markdown <hr>
# @markdown <em><sub>Enter the resource indicator for the application you are accessing and then click the 'run' button to validate.</em></sub><br><br>
RESOURCE_INDICATOR = '' # @param {"type":"string","placeholder":"Enter the resource indicator"}

# @markdown <br>
# @markdown <hr>
SDK_DEBUG_ENABLED = False # @param {"type":"boolean"}
# @markdown <hr>
# @markdown <br>

import re

# Validate OKTA_DOMAIN format
okta_domain_regex = r"^https:\/\/[a-zA-Z0-9-]+\.(okta|oktapreview|okta-emea)\.com$"
if not re.match(okta_domain_regex, OKTA_DOMAIN):
    raise ValueError(
        f"Invalid OKTA_DOMAIN: '{OKTA_DOMAIN}'. "
        "Expected format: 'https://<your-domain>.<okta|oktapreview|okta-emea>.com'"
    )

# Validate other required variables
if not REDIRECT_URI:
    raise ValueError("REDIRECT_URI cannot be empty.")
if not RESOURCE_INDICATOR:
    raise ValueError("RESOURCE_INDICATOR cannot be empty.")

# Validate auth inputs: either client credentials OR principal + private JWK
has_client_credentials = bool(CLIENT_ID and CLIENT_SECRET)
has_principal_credentials = bool(PRINCIPAL_ID and isinstance(PRIVATE_JWK, dict) and PRIVATE_JWK.get('kid'))

if CLIENT_ID and not CLIENT_SECRET:
    raise ValueError("CLIENT_SECRET is required when CLIENT_ID is provided.")
if CLIENT_SECRET and not CLIENT_ID:
    raise ValueError("CLIENT_ID is required when CLIENT_SECRET is provided.")

if PRINCIPAL_ID and (not isinstance(PRIVATE_JWK, dict) or not PRIVATE_JWK.get('kid')):
    raise ValueError("PRIVATE_JWK with at least a 'kid' is required when PRINCIPAL_ID is provided.")
if PRIVATE_JWK and not PRINCIPAL_ID:
    raise ValueError("PRINCIPAL_ID is required when PRIVATE_JWK is provided.")

if not (has_client_credentials or has_principal_credentials):
    raise ValueError(
        "Provide either (CLIENT_ID and CLIENT_SECRET) or (PRINCIPAL_ID and PRIVATE_JWK with 'kid')."
    )

print("✅ All configuration variables validated successfully!")
print(f"Okta Domain: {OKTA_DOMAIN}")
print(f"Client ID: {CLIENT_ID[:20]}..." if len(CLIENT_ID) > 20 else f"Client ID: {CLIENT_ID}")
_kid = PRIVATE_JWK.get('kid')
print(f"KID: {_kid[:20]}..." if _kid and len(_kid) > 20 else f"KID: {_kid}")

global ISSUER
ISSUER = f"{OKTA_DOMAIN}/oauth2"

if (SDK_DEBUG_ENABLED):
	print("SDK debug mode is enabled. Loading debugger... Additional logs will be printed.")
	import requests

	url = "https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/utils.py"

	response = requests.get(url)
	if response.status_code == 200:
		# This executes the code inside utils.py
		exec(response.text)
		print("Successfully loaded utils.py from GitHub!")
	else:
		print("Failed to load utils.py. Check your URL.")


### 3. Initialize the SDK for _user authentication_

In [ ]:
# Initialize Okta SDK
import asyncio
from okta_client.authfoundation import OAuth2Client, OAuth2ClientConfiguration, ClientSecretAuthorization, LocalKeyProvider
from okta_client.authfoundation.oauth2.jwt_bearer_claims import JWTBearerClaims
from okta_client.authfoundation.oauth2.client_authorization import ClientAssertionAuthorization
from okta_client.oauth2auth import AuthorizationCodeContext, AuthorizationCodeFlow, CrossAppAccessFlow, CrossAppAccessTarget, Prompt

print("✅ Imports successful!")

global user_client_authorization

global KEY_PROVIDER
# Create key provider
# If you are using client credentials, this will still be used later to initialize the ClientAssertionAuthorization for the agent.
KEY_PROVIDER = LocalKeyProvider(
    key=PRIVATE_JWK,
    algorithm=PRIVATE_JWK.get('alg', 'RS256'),
    key_id=PRIVATE_JWK.get('kid')
)

global JWT_CLAIMS
# Create JWT bearer claims
# If you are using client credentials, this will still be used later to initialize the ClientAssertionAuthorization for the agent.
JWT_CLAIMS = JWTBearerClaims(
    issuer=PRINCIPAL_ID,
    subject=PRINCIPAL_ID,
    audience=f"{OKTA_DOMAIN}/oauth2/v1/token",
    expires_in=300
)

if (has_client_credentials):

    print("\n CLIENT_ID and CLIENT_SECRET provided. Using ClientSecretAuthorization for user SDK...")

    user_client_authorization = ClientSecretAuthorization(
        id=CLIENT_ID,
        secret=CLIENT_SECRET
    )
else:
    print("\n PRINCIPAL_ID and PRIVATE_JWK provided. Using ClientAssertionAuthorization for user SDK...")
    print("✅ Key provider and JWT claims created for ClientAssertionAuthorization.")

    user_client_authorization = ClientAssertionAuthorization(
        assertion_claims=JWT_CLAIMS,
        key_provider=KEY_PROVIDER
    )

user_sdk_config = OAuth2ClientConfiguration(
    issuer=OKTA_DOMAIN,
    scope=["openid", "profile"],
    redirect_uri=REDIRECT_URI,
    client_authorization=user_client_authorization
)

user_sdk = OAuth2Client(configuration=user_sdk_config)

print("✅ User SDK initialized!")

if SDK_DEBUG_ENABLED:
    debugger_cls = globals().get("Debugger")
    if debugger_cls is None:
        print("[WARN] Debugger not loaded; continuing without SDK listener.")
    else:
        user_sdk.listeners.add(debugger_cls()) # type: ignore


### 4. Initialize the Okta _agent_ SDK

First, let's initialize a new instance of the SDK with cross-app access configuration for our agent.

In [ ]:
# Create key provider
if not (KEY_PROVIDER and isinstance(KEY_PROVIDER, LocalKeyProvider)):

    KEY_PROVIDER = LocalKeyProvider(
        key=PRIVATE_JWK,
        algorithm=PRIVATE_JWK.get('alg', 'RS256'),
        key_id=PRIVATE_JWK.get('kid')
    )
    print("✅ Key provider created")

if not (JWT_CLAIMS and isinstance(JWT_CLAIMS, JWTBearerClaims)):
    # Create JWT bearer claims
    JWT_CLAIMS = JWTBearerClaims(
        issuer=PRINCIPAL_ID,
        subject=PRINCIPAL_ID,
        audience=f"{OKTA_DOMAIN}/oauth2/v1/token",
        expires_in=300
    )
    print("✅ JWT bearer claims created")

agent_sdk_config = OAuth2ClientConfiguration(
    issuer=OKTA_DOMAIN,
    client_authorization=ClientAssertionAuthorization(
        assertion_claims=JWT_CLAIMS,
        key_provider=KEY_PROVIDER,
    )
)

print("✅ OAuth2 client configuration created")

if SDK_DEBUG_ENABLED:
    debugger_cls = globals().get("Debugger")
    if debugger_cls is None:
        print("[WARN] Debugger not loaded; continuing without SDK listener.")
    else:
        agent_sdk.listeners.add(debugger_cls()) # type: ignore

from okta_client.authfoundation import APIClientListener, APIRetry

# The following code is used to intercept the interaction_uri in order to trigger the user consent flow.
# This is necessary until the SDK is enhanced to handle this natively.
class RequestLogger(APIClientListener):

    def will_send(self, client, request):
        # print(f"→ {request.method.value} {request.url}")
        pass

    def did_send(self, client, request, response):
        # print(f"← {response.status_code}")
        result = getattr(response, 'result', None)

        interaction_uri = (
            result.get('interaction_uri') if isinstance(result, dict)
            else getattr(result, 'interaction_uri', None)
        )

        if interaction_uri:
            global INTERACTION_URI
            INTERACTION_URI = interaction_uri
            # print(f"   Interaction URI: {INTERACTION_URI}")

    def did_send_error(self, client, request, error):
        print(f"✗ {error}")
        pass

    def should_retry(self, client, request, rate_limit):
        return APIRetry.default()

print("✓ OAuth2 client configuration created")

# Create OAuth2 client
global agent_sdk
agent_sdk = OAuth2Client(configuration=agent_sdk_config)
agent_sdk.listeners.add(RequestLogger())

print("✅ OAuth2 client created")

print("✅ Agent SDK initialized successfully!")

---

## STEP 1: Obtain ID Token

If you don't have an ID token yet, use this section to obtain one through OAuth 2.0 authorization code flow.

### Important:
This step uses the **Org Authorization Server** (not a custom authorization server). The ID token will be issued directly by your Okta domain:
- Issuer: `https://your-domain.okta.com`
- Endpoint: `/oauth2/v1/authorize` and `/oauth2/v1/token`

This is the correct approach for obtaining the initial user ID token that will be used in the ID-JAG flow.

### The Flow:

1. **Build authorization URL** - User authenticates in browser (Org Authorization Server)
1. **Copy redirect URL from browser** - Copy the entire redirect URL after authentication.
1. **Exchange code for tokens** - Get ID token with issuer = Okta domain

### 1. Build and Open Authorization URL

Run the following cell and then click the generated button to open the authorization URL in a new browser tab.

In [ ]:
from IPython.display import HTML, display # Ensure display is imported for the HTML button

# ID Token (will be obtained in STEP 0, or provide your own)
ID_TOKEN = None  # Leave as None to obtain via authorization code flow
ACCESS_TOKEN = None
REFRESH_TOKEN = None

async def authorize():

  try:
    # Step 1: Build the authorization URL
    authorization_context = AuthorizationCodeContext(
        prompt=Prompt.LOGIN,
    )

    global auth_flow
    auth_flow = AuthorizationCodeFlow(client=user_sdk)

    authorization_url = await auth_flow.start(context=authorization_context)

    print("Authorization URL generated!")
    print(f"\n{authorization_url}")
    print("\n" + "="*80)

    # Display clickable button
    html_button = f"""
    <div style="margin: 20px 0;">
        <a href="{authorization_url}" target="_blank" style="
            display: inline-block;
            padding: 15px 30px;
            background-color: #007bff;
            color: white;
            text-decoration: none;
            border-radius: 5px;
            font-weight: bold;
            font-size: 16px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.2);
        ">Click Here to Authenticate with Okta</a>
    </div>
    <div style="margin: 20px 0; padding: 15px; background-color: #fff3cd; border-left: 4px solid #ffc107; border-radius: 4px;">
        <strong>Instructions:</strong>
        <ol style="margin: 10px 0 0 0;">
            <li>Click the button above to open the authorization URL in a new tab</li>
            <li>Sign in with your Okta credentials</li>
            <li>After authentication, you'll be redirected to: <code>{REDIRECT_URI}</code> which will fail</li>
            <li>Copy the entire URL of failed redirect from browser URL bar
            <li>Paste the code in the next cell to exchange it for tokens</li>
        </ol>
    </div>
    """

    display(HTML(html_button))

    print("\nWhat to do next:")
    print("   1. Click the button above")
    print("   2. Sign in to Okta")
    print("   3. Copy the entire URL from failed redirect in browser")
    print("   4. Paste it in the next cell")
    print("\nNote: The ID token will be issued by the Org Authorization Server")
    print(f"      Issuer: {OKTA_DOMAIN}")

  except Exception as e:

    print(f"❌ [ERROR]: {e}")
    raise

await authorize()

### 2. Exchange Authorization Code for Tokens

After authenticating...
1. copy the entire URL
1. paste the URL below
1. and run this cell to obtain tokens

In [ ]:
# @title { display-mode: "form" }
REDIRECT_URL = '' # @param { type: "string", placeholder: "Insert entire URL string here."}

async def exchange_code_for_tokens():
  if not REDIRECT_URL:
    print("\nNo url provided! Please paste the entire URL containing the `code` and run this cell again.")
  else:
    print("\n" + "=" * 80)
    print("STEP 1: Exchange Access Code for User Tokens")
    print("=" * 80)

    try:
      # Step 2: Exchange the authorization code for tokens
      redirect_url = REDIRECT_URL

      token = await auth_flow.resume(redirect_url)

      print("Token exchange successful!\n")
      print("Token Response:")
      print(f"   Token Type: {token.token_type}")
      print(f"   Expires In: {token.expires_in} seconds")
      print(f"   Scope: {token.scope}")

      # Extract tokens
      global ID_TOKEN, ACCESS_TOKEN, REFRESH_TOKEN
      ID_TOKEN = token.id_token.raw
      ACCESS_TOKEN = token.access_token
      REFRESH_TOKEN = token.refresh_token

      print(f"\nTokens Obtained:")
      if ID_TOKEN:
          print(f" ✅ ID Token: https://jwt.io#token={ID_TOKEN}")
      if ACCESS_TOKEN:
          print(f" ✅ Access Token: https://jwt.io#token={ACCESS_TOKEN}")
      if REFRESH_TOKEN:
          print(f" ✅ Refresh Token: {REFRESH_TOKEN[:50]}...")

      # Decode and display ID token claims (optional)
      if ID_TOKEN:
          import jwt
          decoded = jwt.decode(ID_TOKEN, options={"verify_signature": False})
          print(f"\nID Token Claims:")
          print(f"   Subject: {decoded.get('sub')}")
          print(f"   Email: {decoded.get('email', 'N/A')}")
          print(f"   Name: {decoded.get('name', 'N/A')}")
          print(f"   Issuer: {decoded.get('iss')}")
          print(f"   Audience: {decoded.get('aud')}")

          # Verify issuer is the Okta domain (Org Authorization Server)
          if decoded.get('iss') == OKTA_DOMAIN:
              print(f"\n   [OK] Token issued by Org Authorization Server: {OKTA_DOMAIN}")
          else:
              print(f"\n   [WARNING] Unexpected issuer: {decoded.get('iss')}")
              print(f"   Expected: {OKTA_DOMAIN}")

      print("Now verifying configuration...")

      # Verify configuration is set
      print("Configuration")
      print("=" * 60)

      if ID_TOKEN and ID_TOKEN != "your_id_token_here" and ID_TOKEN is not None:
          print(f"[OK] Using ID Token from STEP 0: {ID_TOKEN[:30]}...")
      else:
          print("[WARNING] No ID token set. Please run previous step again or set ID_TOKEN manually.")
          print("          The ID-JAG flow will fail without a valid ID token.")

      print(f"\nConfiguration Summary:")
      print(f"   Okta Domain: {OKTA_DOMAIN}")

      if (CLIENT_ID and CLIENT_SECRET):
              print(f"   Client ID: {CLIENT_ID[:20]}..." if len(CLIENT_ID) > 20 else f"   Client ID: {CLIENT_ID}")

      print(f"   Principal ID: {PRINCIPAL_ID[:20]}..." if len(PRINCIPAL_ID) > 20 else f"   Principal ID: {PRINCIPAL_ID}")
      print(f"   Private JWK: {'Configured' if PRIVATE_JWK.get('kid') != 'your_key_id' else 'Placeholder'}")
      print("=" * 60)

    except Exception as e:
        print(f" ❌ Error during token exchange: {e}")
        print("\nTroubleshooting:")
        print("   • Make sure you copied the entire authorization code")
        print("   • Verify your redirect_uri matches what's registered in Okta")
        print("   • Check that the authorization code hasn't expired (valid for ~60 seconds)")
        print("   • Ensure your client_id and client_secret are correct")

await exchange_code_for_tokens()

---

---
## Step 3: Exchange user token for resource access token

This step performs two actions:
1. Refreshes the user token set when a `REFRESH_TOKEN` is available.
2. Exchanges the user's token for the final access token using your configured `RESOURCE_INDICATOR`.

### Prerequisites
- Setup and Step 1 have been run at least once.
- User's token and `agent_sdk` are available in memory.
- `RESOURCE_INDICATOR` points to the resource connected in Okta.

### Output
- Stores the exchanged resource access token in `RESOURCE_TOKEN`.

In [ ]:
from IPython.display import HTML, display
from okta_client.authfoundation.oauth2.refresh_token import RefreshTokenFlow
from okta_client.oauth2auth.token_exchange import TokenExchangeFlow, TokenType

global RESOURCE_TOKEN
RESOURCE_TOKEN = None

def _require_state(name: str):
    if name not in globals() or globals().get(name) is None:
        raise ValueError(f"Missing required value: {name}. Run earlier setup/authentication steps first.")

async def refresh_tokens_if_available():
    print("\n" + "=" * 80)
    print("Step 3A: Ensure fresh user token")
    print("=" * 80)

    _require_state("ID_TOKEN")
    refresh_token = globals().get("REFRESH_TOKEN")

    if not refresh_token:
        print("No REFRESH_TOKEN available. Continuing with current ID_TOKEN.")
        print("If this token is expired, rerun Step 1 to get a fresh ID token.")
        return globals()["ID_TOKEN"]

    try:
        refreshed_result = await RefreshTokenFlow(client=user_sdk).start(refresh_token)
        globals()["ID_TOKEN"] = refreshed_result.id_token.raw
        globals()["REFRESH_TOKEN"] = refreshed_result.refresh_token

        print("✅ Tokens refreshed successfully")
        print(f"   Expires In: {refreshed_result.expires_in} seconds")
        print(f"   Scope: {refreshed_result.scope or 'N/A'}")
        return globals()["ID_TOKEN"]
    except Exception as e:
        print(f"[WARNING] Refresh failed: {e}")
        print("Continuing with existing ID_TOKEN. If exchange fails, rerun Step 1.")
        return globals()["ID_TOKEN"]

async def exchange_for_resource_token(id_token: str):
    print("\n" + "=" * 80)
    print("Step 3B: Exchange ID token for resource access token")
    print("=" * 80)

    _require_state("agent_sdk")

    try:
        flow = TokenExchangeFlow(client=agent_sdk)
        token_result = await flow.start(
            subject_token=globals()["ID_TOKEN"],
            subject_token_type=TokenType.ID_TOKEN,
            resource=[RESOURCE_INDICATOR],
            requested_token_type="urn:okta:params:oauth:token-type:oauth-sts",
        )

        print("✅ Resource access token exchange successful")
        print(f"   Token Type: {token_result.token_type}")
        print(f"   Expires In: {token_result.expires_in} seconds")
        print(f"   Scope: {token_result.scope or 'N/A'}")

        return token_result.access_token

    except Exception as e:
        print(f"[ERROR] Token exchange failed: {e}")

        if "INTERACTION_URI" in globals() and INTERACTION_URI:
            print("Additional user interaction is required. Authorize, then rerun Step 3B.")

            html_button = f"""
            <div style=\"margin: 20px 0;\">
                <a href=\"{INTERACTION_URI}\" target=\"_blank\" style=\"
                    display: inline-block;
                    padding: 15px 30px;
                    background-color: #007bff;
                    color: white;
                    text-decoration: none;
                    border-radius: 5px;
                    font-weight: bold;
                    font-size: 16px;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.2);
                \">Click Here to Authorize your Resource</a>
            </div>
            <div style=\"margin: 20px 0; padding: 15px; background-color: #fff3cd; border-left: 4px solid #ffc107; border-radius: 4px;\">
                <strong>Instructions:</strong>
                <ol style=\"margin: 10px 0 0 0;\">
                    <li>Click the button above to open authorization in a new tab.</li>
                    <li>Complete sign-in/consent.</li>
                    <li>Return here and rerun Step 3B.</li>
                </ol>
            </div>
            """
            display(HTML(html_button))
            return None

        raise

current_id_token = await refresh_tokens_if_available()
RESOURCE_TOKEN = await exchange_for_resource_token(current_id_token)

if RESOURCE_TOKEN:
    print("   RESOURCE_TOKEN is now set for Step 4")
    print(f"   Token preview (if token is a JWT): https://jwt.io#token={RESOURCE_TOKEN}")

---
## Step 4: Use Resource Access Token to Call Resource API

This step uses `RESOURCE_TOKEN` from Step 3 to call your resource's endpoint.

_You will need to configure the endpoint in order for this step to work._

### Hard-fail behavior
- If `RESOURCE_TOKEN` is missing, this step fails with remediation instructions.
- If the resource returns `401`, rerun Step 3 to obtain a fresh final resource access token.
- If the resource returns `403`, verify the connected app granted the required email scope.

In [ ]:
import http.client
import json

# @title { display-mode: "form" }
# @markdown <hr>
# @markdown <br><em>You may need to modify this code in order to call your specific resource.</em>
# @markdown <br><br><em>We have provided some sample code for each resource type outlined in the sample guide. Adjust the code accordingly.</em><br><br>
# @markdown <br>
# @markdown <hr>

global RESOURCE_PROVIDER
global RESOURCE_URL
global RESOURCE_ENDPOINT
global RESOURCE_BASE_URL

# @markdown Select provider mode. Use <b>auto</b> to infer from filled fields.
RESOURCE_PROVIDER = 'auto' # @param ["auto", "custom", "atlassian", "snow", "github", "msft", "slack"]
RESOURCE_URL = '' # @param {"type":"string","placeholder":"Enter the resource base URL"}
RESOURCE_ENDPOINT = '' # @param {"type":"string","placeholder":"Enter the resource path URL."}

# @markdown <hr>
# @markdown Atlassian Jira
ATLASSIAN_SITE_URL = '' # @param {"type":"string","placeholder":"Enter your Atlassian site URL (e.g., yoursite.atlassian.net)"}
ATLASSIAN_ENDPOINT = '/rest/api/3/project' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

# @markdown <hr>
# @markdown ServiceNow
SNOW_INSTANCE = '' # @param {"type":"string","placeholder":"Enter your ServiceNow instance name (e.g., dev12345)"}
SNOW_URL = f'{SNOW_INSTANCE}.service-now.com' # @param {"type":"string","placeholder":"Enter the resource base URL"}
SNOW_ENDPOINT = '/api/now/table/incident' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

# @markdown <hr>
# @markdown Github
GITHUB_URL = 'api.github.com' # @param {"type":"string","placeholder":"Enter your Github URL (if different)"}
GITHUB_ENDPOINT = '/user/emails' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

# @markdown <hr>
# @markdown Office 365
MSFT_URL = 'graph.microsoft.com' # @param {"type":"string","placeholder":"Enter the resource base URL"}
MSFT_ENDPOINT = '/v1.0/me' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

# @markdown <hr>
# @markdown Slack
SLACK_URL = 'slack.com' # @param {"type":"string","placeholder":"Enter the resource base URL"}
SLACK_ENDPOINT = '/api/conversations.list' # @param {"type":"string","placeholder":"Enter the resource endpoint URL"}

if not RESOURCE_TOKEN:
    raise ValueError(
        "RESOURCE_TOKEN is missing. Rerun Step 3 to complete the resource access token exchange."
    )

def _normalize_url(url: str) -> str:
    return (url or '').strip().replace('https://', '').replace('http://', '').rstrip('/')

def _normalize_endpoint(endpoint: str) -> str:
    endpoint = (endpoint or '').strip()
    if endpoint and not endpoint.startswith('/'):
        endpoint = f"/{endpoint}"
    return endpoint

provider = (RESOURCE_PROVIDER or 'auto').strip().lower()
allowed = {'auto', 'custom', 'atlassian', 'snow', 'github', 'msft', 'slack'}
if provider not in allowed:
    raise ValueError(f"Invalid RESOURCE_PROVIDER '{RESOURCE_PROVIDER}'. Must be one of: {sorted(allowed)}")

resolved_from = None
resolved_url = ''
resolved_endpoint = ''

if provider == 'custom':
    resolved_url = RESOURCE_URL
    resolved_endpoint = RESOURCE_ENDPOINT
    resolved_from = 'custom provider selection'
elif provider == 'atlassian':
    resolved_url = ATLASSIAN_SITE_URL
    resolved_endpoint = RESOURCE_ENDPOINT or ATLASSIAN_ENDPOINT
    resolved_from = 'Atlassian provider selection'
elif provider == 'snow':
    resolved_url = SNOW_URL
    resolved_endpoint = RESOURCE_ENDPOINT or SNOW_ENDPOINT
    resolved_from = 'ServiceNow provider selection'
elif provider == 'github':
    resolved_url = GITHUB_URL
    resolved_endpoint = RESOURCE_ENDPOINT or GITHUB_ENDPOINT
    resolved_from = 'Github provider selection'
elif provider == 'msft':
    resolved_url = MSFT_URL
    resolved_endpoint = RESOURCE_ENDPOINT or MSFT_ENDPOINT
    resolved_from = 'MSFT provider selection'
elif provider == 'slack':
    resolved_url = SLACK_URL
    resolved_endpoint = RESOURCE_ENDPOINT or SLACK_ENDPOINT
    resolved_from = 'Slack provider selection'
else:
    # Auto priority: explicit custom -> Atlassian -> ServiceNow -> Github
    if RESOURCE_URL and RESOURCE_ENDPOINT:
        resolved_url = RESOURCE_URL
        resolved_endpoint = RESOURCE_ENDPOINT
        resolved_from = 'auto: explicit RESOURCE_URL/RESOURCE_ENDPOINT'
    elif ATLASSIAN_SITE_URL:
        resolved_url = ATLASSIAN_SITE_URL
        resolved_endpoint = RESOURCE_ENDPOINT or ATLASSIAN_ENDPOINT
        resolved_from = 'auto: Atlassian settings'
    elif SNOW_INSTANCE:
        resolved_url = SNOW_URL
        resolved_endpoint = RESOURCE_ENDPOINT or SNOW_ENDPOINT
        resolved_from = 'auto: ServiceNow settings'
    else:
        resolved_url = GITHUB_URL
        resolved_endpoint = RESOURCE_ENDPOINT or GITHUB_ENDPOINT
        resolved_from = 'auto: Github defaults'

RESOURCE_URL = _normalize_url(resolved_url)
RESOURCE_ENDPOINT = _normalize_endpoint(resolved_endpoint)

if not RESOURCE_URL or not RESOURCE_ENDPOINT:
    raise ValueError(
        "Unable to resolve resource target. Set RESOURCE_PROVIDER to a specific provider, "
        "or provide RESOURCE_URL and RESOURCE_ENDPOINT for custom mode."
    )

RESOURCE_BASE_URL = RESOURCE_URL  # Backward-compatible alias

print("\n" + "=" * 80)
print(f"STEP 4: Use resource access token to call {RESOURCE_ENDPOINT}")
print("=" * 80)
print(f"Resolved target from: {resolved_from}")
print(f"Resolved base URL: {RESOURCE_URL}")
print(f"Making API call to resource endpoint {RESOURCE_ENDPOINT}...")

conn = http.client.HTTPSConnection(RESOURCE_URL)
headers = {
    "Accept": "application/json",
    "User-Agent": "okta-ai-poc/1.0",
    "Authorization": f"Bearer {RESOURCE_TOKEN}"
}

if 'api.github.com' in RESOURCE_URL:
    headers['X-GitHub-Api-Version'] = '2022-11-28'

conn.request("GET", RESOURCE_ENDPOINT, "", headers)
res = conn.getresponse()
raw_data = res.read().decode("utf-8")

print(f"Response Status: {res.status} {res.reason}")

if res.status >= 400:
    detail = raw_data
    try:
        detail = json.dumps(json.loads(raw_data), indent=2)
    except Exception:
        pass

    remediation = "Verify the connected app granted the required permissions/scopes for this API."
    if res.status == 401:
        remediation = "Resource rejected the token. Rerun Step 3 to obtain a fresh final access token."

    raise RuntimeError(
        "Resource API call failed. "
        f"{remediation} "
        f"Status={res.status}. Response={detail}"
    )

try:
    parsed = json.loads(raw_data)
except Exception:
    parsed = {"raw": raw_data}

print("Response from resource API:")
print(json.dumps(parsed, indent=2))

---

## Resources

- [Okta AI SDK Documentation](https://github.com/okta/okta-client-python/tree/main)

---
